# 综合实训 · 通用矩阵乘法 GEMM  (C = A · B)

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐⭐⭐⭐ 综合　|　**预计时长**：60–90 分钟

> **实验说明**
> 1. 本实验是第三章的**综合实训**：由 **v1 串行基准**起，依次引入 **v2 朴素 NEON**、**v3 寄存器分块**、**v4 Cache 分块**、**v5 内存打包**，每一版本均实际编译并运行，据此观察性能的逐步变化。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 五个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方法，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。
> 6. 本实验建立在 **实验一 AXPY**（逐元素运算）与 **实验二 GEMV**（规约、寄存器分块）之上，建议先完成这两个实验。
> 7. ⏱️ 默认规模为 1024×1024×1024。串行基准单次即需数秒，故其重复次数单独设为 `NTIMES_BASE = 1`；五个版本全部跑完约需 2–4 分钟。


## 🎯 学习目标

完成本实验后，学生应能够：

- 理解 **BLAS Level-3** 与 Level-1/2 的本质区别：GEMM 的计算量 $O(n^3)$ 高于访存量 $O(n^2)$，是**唯一有可能逼近算力屋顶**的一类核心运算
- 掌握**算术强度**的定量计算方法，并能由微内核尺寸 $m\times n$ 直接推出 $I=\dfrac{mn}{2(m+n)}$
- 理解"**向量化的方向选择**"：为什么手写 NEON 沿 $j$ 方向向量化，而编译器倾向于沿 $k$（规约）方向，以及后者为何几乎必然更慢
- 掌握 **寄存器分块**（4×4 微内核）、**Cache 分块**（loop tiling）与 **内存打包**（packing）三级优化，理解它们分别针对存储层次的哪一级
- 理解沿 K 方向分块后，微内核必须由"覆盖写"改为"**累加写**"，以及由此带来的 C 块清零要求
- 能够独立完成 **8×8 双打包微内核**（扩展实验），并用寄存器数量约束解释微内核尺寸的选择上限


## 🗺️ 学习路径

1. **准备阶段**：理解 GEMM 的定义、计算量，以及"算术强度"这把贯穿全实验的尺子
2. **v1 · 串行基准**：实现 `gemm_serial_no_vec`（关闭向量化，作为基准）与 `gemm_serial`（允许自动向量化）
   → 考察编译器对**三重循环**的自动向量化能力
3. **v2 · 朴素 NEON**：新增 `gemm_neon_naive`，1×4 微内核，沿 $j$ 方向向量化
   → 掌握"广播 A、加载 B、FMA 累加"这一 GEMM 的基本向量化范式
4. **v3 · 寄存器分块**：新增 `gemm_neon_reg4x4`，4×4 微内核，4 个累加器常驻寄存器
   → 考察提高数据复用（算术强度由 0.4 升到 1.0）的效果
5. **v4 · Cache 分块**：新增 `gemm_neon_tiled`，把 M/N/K 三个方向都切成 64 的小块
   → 考察让工作集驻留 L1D/L2 的效果，并理解"累加写"的必要性
6. **v5 · 内存打包**：新增 `gemm_neon_packed`，把 B 块复制到连续缓冲区
   → 考察消除跨页跳跃、改善 TLB 与 cache line 利用率的效果
7. **可视化与分析**：绘制加速比与 GFLOPS 柱状图，用算术强度与汇编证据解释每一级的收益来源
8. **🚀 扩展实验**：独立完成 8×8 双打包微内核，把算术强度进一步提高到 2.0


## 1. 背景与动机

GEMM（General Matrix Multiply）是 BLAS **Level-3** 操作：矩阵乘矩阵，`C = A · B`。它在数值线性代数、科学计算与深度学习中无处不在——卷积、全连接层、注意力机制在底层几乎都被归约为 GEMM。**GEMM 的性能，很大程度上就是一台机器"能算多快"的实际上限。**

GEMM 与前两个实验有一个根本性的区别：

<!--
| 运算 | 级别 | 计算量 | 访存量 | 计算/访存之比 |
|---|---|---|---|---|
| AXPY `y = a·x + y` | Level-1 | $O(n)$ | $O(n)$ | $O(1)$ —— 恒定，**必然受访存制约** |
| GEMV `y = A·x` | Level-2 | $O(n^2)$ | $O(n^2)$ | $O(1)$ —— 恒定，**必然受访存制约** |
| **GEMM `C = A·B`** | **Level-3** | $O(n^3)$ | $O(n^2)$ | $O(n)$ —— **随规模增长** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">运算</th>
      <th style="text-align: left;">级别</th>
      <th style="text-align: left;">计算量</th>
      <th style="text-align: left;">访存量</th>
      <th style="text-align: left;">计算/访存之比</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">AXPY <code>y = a·x + y</code></td>
      <td style="text-align: left;">Level-1</td>
      <td style="text-align: left;">O(n)</td>
      <td style="text-align: left;">O(n)</td>
      <td style="text-align: left;">O(1) —— 恒定，<strong>必然受访存制约</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">GEMV <code>y = A·x</code></td>
      <td style="text-align: left;">Level-2</td>
      <td style="text-align: left;">O(n<sup>2</sup>)</td>
      <td style="text-align: left;">O(n<sup>2</sup>)</td>
      <td style="text-align: left;">O(1) —— 恒定，<strong>必然受访存制约</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>GEMM <code>C = A·B</code></strong></td>
      <td style="text-align: left;"><strong>Level-3</strong></td>
      <td style="text-align: left;">O(n<sup>3</sup>)</td>
      <td style="text-align: left;">O(n<sup>2</sup>)</td>
      <td style="text-align: left;">O(n) —— <strong>随规模增长</strong></td>
    </tr>
  </tbody>
</table>

前两个实验中，无论怎样优化，最终都会撞上内存带宽这堵墙。GEMM 不同：**它的计算量比访存量高一个数量级，理论上完全有可能把数据留在寄存器和 Cache 里反复使用，从而让 CPU 的浮点单元一直满负荷运转。**

但"理论上可能"不等于"写出来就有"。朴素的三重循环只能跑出峰值算力的百分之几。本实验要回答的就是：**这中间差的几十倍，是怎样一步一步补回来的。**


## 2. 算法与公式

$$C_{ij}=\sum_{k=0}^{K-1} A_{ik}\,B_{kj}$$

浮点运算量为 $2MNK$（每个 $C_{ij}$ 需要 $K$ 次乘法与 $K$ 次加法）。最朴素的实现就是三重循环：

```c
for (i = 0; i < M; i++)
  for (j = 0; j < N; j++) {
    float sum = 0.0f;
    for (k = 0; k < K; k++) sum += A[i*K+k] * B[k*N+j];
    C[i*N+j] = sum;
  }
```

三层循环可以任意交换次序，也可以任意分块——**这正是 GEMM 优化空间巨大的原因**。但朴素写法有一个致命问题：内层 `k` 循环中，`A[i*K+k]` 是连续访问（步长 1），而 `B[k*N+j]` 的步长是 `N`。当 N = 1024 时，相邻两次访问的地址相差 4 KiB，**每一次都落在不同的 Cache line、甚至不同的物理页上**。

> **关键认识**：GEMM 的困难不在"算"，而在"喂"。整个实验的五个版本，本质上都在做同一件事——**让每一个从内存搬进来的数，被使用尽可能多次**。


## 3. 核心概念：算术强度与微内核

### 3.1 算术强度（Arithmetic Intensity）

算术强度定义为「每从内存搬运 1 字节数据，能完成多少次浮点运算」：

$$I=\frac{\text{FLOP}}{\text{Byte}}$$

设微内核一次计算 C 的一个 $m\times n$ 子块。内层每推进一个 $k$：

- **搬入**：A 的 $m$ 个标量 + B 的 $n$ 个元素 = $4(m+n)$ 字节
- **算出**：$m\times n$ 次乘加 = $2mn$ FLOP

$$\boxed{\;I=\frac{2mn}{4(m+n)}=\frac{mn}{2(m+n)}\;\text{FLOP/Byte}\;}$$

<!--
| 微内核 | 算术强度 $I$ | 本实验中的版本 |
|---|---|---|
| 1×1（标量） | 0.25 | v1 串行基准 |
| **1×4** | **0.40** | v2 朴素 NEON |
| **4×4** | **1.00** | v3 寄存器分块 |
| **8×8** | **2.00** | 🚀 扩展实验 |
| 8×12 | 2.40 | （OpenBLAS 在部分 ARM 平台上的选择） |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">微内核</th>
      <th style="text-align: left;">算术强度 <i>I</i></th>
      <th style="text-align: left;">本实验中的版本</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">1×1（标量）</td>
      <td style="text-align: left;">0.25</td>
      <td style="text-align: left;">v1 串行基准</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>1×4</strong></td>
      <td style="text-align: left;"><strong>0.40</strong></td>
      <td style="text-align: left;">v2 朴素 NEON</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>4×4</strong></td>
      <td style="text-align: left;"><strong>1.00</strong></td>
      <td style="text-align: left;">v3 寄存器分块</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>8×8</strong></td>
      <td style="text-align: left;"><strong>2.00</strong></td>
      <td style="text-align: left;">🚀 扩展实验</td>
    </tr>
    <tr>
      <td style="text-align: left;">8×12</td>
      <td style="text-align: left;">2.40</td>
      <td style="text-align: left;">（OpenBLAS 在部分 ARM 平台上的选择）</td>
    </tr>
  </tbody>
</table>

这个公式说明了一件重要的事：**$m$ 和 $n$ 应当尽量接近且尽量大**。1×16 的强度只有 0.94，而 4×4 就有 1.00——同样是 16 个累加元素，方形子块的复用率更高。

### 3.2 上限从哪来：寄存器数量

AArch64 共有 **32 个 128 位向量寄存器**（v0–v31），每个装 4 个 float。一个 $m\times n$ 微内核至少需要：

$$\underbrace{\frac{mn}{4}}_{\text{累加器}}+\underbrace{\frac{n}{4}}_{\text{B 向量}}+\underbrace{\text{若干}}_{\text{A 标量/临时}}\;\le\;32$$

<!--
| 微内核 | 累加器 | B 向量 | 合计 | 是否可行 |
|---|---|---|---|---|
| 4×4 | 4 | 1 | 5 | ✅ 余量很大（未用满） |
| 8×8 | 16 | 2 | 18 | ✅ 舒适 |
| 8×12 | 24 | 3 | 27 | ⚠️ 紧张 |
| 16×16 | 64 | 4 | 68 | ❌ 必然溢出到栈 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">微内核</th>
      <th style="text-align: left;">累加器</th>
      <th style="text-align: left;">B 向量</th>
      <th style="text-align: left;">合计</th>
      <th style="text-align: left;">是否可行</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">4×4</td>
      <td style="text-align: left;">4</td>
      <td style="text-align: left;">1</td>
      <td style="text-align: left;">5</td>
      <td style="text-align: left;">✅ 余量很大（未用满）</td>
    </tr>
    <tr>
      <td style="text-align: left;">8×8</td>
      <td style="text-align: left;">16</td>
      <td style="text-align: left;">2</td>
      <td style="text-align: left;">18</td>
      <td style="text-align: left;">✅ 舒适</td>
    </tr>
    <tr>
      <td style="text-align: left;">8×12</td>
      <td style="text-align: left;">24</td>
      <td style="text-align: left;">3</td>
      <td style="text-align: left;">27</td>
      <td style="text-align: left;">⚠️ 紧张</td>
    </tr>
    <tr>
      <td style="text-align: left;">16×16</td>
      <td style="text-align: left;">64</td>
      <td style="text-align: left;">4</td>
      <td style="text-align: left;">68</td>
      <td style="text-align: left;">❌ 必然溢出到栈</td>
    </tr>
  </tbody>
</table>

> **寄存器一旦不够，编译器就会把累加器溢出（spill）到栈上**——每个 k 都要 `str`/`ldr` 一遍，收益立刻被吃光。这是微内核尺寸的硬约束。本实验的 v3 与扩展实验可以用 `gcc -O3 -S` 亲自验证（见动手练习）。

### 3.3 核心 NEON 指令

```c
float32x4_t c_0 = vdupq_n_f32(0.0f);            // 累加器清零
float32x4_t b_vec = vld1q_f32(&B[k * N + j]);   // 加载 B 的 4 个相邻元素
float32x4_t a_val = vdupq_n_f32(A[i * K + k]);  // 【广播】A 的 1 个标量到 4 条通道
c_0 = vfmaq_f32(c_0, a_val, b_vec);             // 乘加融合：c += a * b
c_0 = vfmaq_n_f32(c_0, b_vec, A[i * K + k]);    // 等价写法，直接吃标量（更简洁）
vst1q_f32(&C[i * N + j], c_0);                  // 写回 4 个结果
```

- **`vdupq_n_f32`**：把一个标量**广播**到向量的 4 条通道。这是 GEMM 与 GEMV 最大的写法差异——GEMV 是"两个向量逐通道相乘再规约"，GEMM 是"一个标量乘一个向量"。
- **`vfmaq_f32` / `vfmaq_n_f32`**：乘加融合。`_n_` 后缀版本直接接受标量参数，编译器会生成 `fmla v.4s, v.4s, v.s[0]`（按 lane 取标量），**无需单独的 `dup` 指令**。
- **注意 GEMM 里没有水平规约**：因为向量的 4 条通道对应的是 C 的 4 个**不同**元素（$j, j{+}1, j{+}2, j{+}3$），彼此独立，最后直接 `vst1q_f32` 写回即可。

> **⭐ 本实验最重要的一个认识：向量化的"方向"**
>
> 三重循环有三个方向可以向量化：
> - 沿 **$k$ 方向**（规约方向）：4 条通道是同一个 $C_{ij}$ 的 4 个部分和，**结尾必须做水平规约**，且 B 的访问步长为 N（跨页）。
> - 沿 **$j$ 方向**：4 条通道是 4 个**不同**的 $C_{ij}$，彼此独立，B 的访问**连续**，无需规约。
>
> 本实验的所有手写版本都沿 **$j$ 方向**向量化。而编译器在 `-ffast-math` 下会选择 $k$ 方向——第 11 节将用汇编说明它为此付出了什么代价。


## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows


def parse_gflops(text):
    """GEMM 专用：额外取出第 4 列 GFLOPS。表格列序为 方法|耗时|加速比|GFLOPS|校验。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 4:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mg = re.search(r"[-+]?\d*\.?\d+", cells[3])
        if not mg:
            continue
        rows.append({"method": name, "gflops": float(mg.group())})
    return rows

In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


def plot_gflops(rows, title=""):
    """GEMM 专用：绘制实际算力（GFLOPS）柱状图。"""
    if not rows:
        print("未解析到可绘制的 GFLOPS。")
        return
    names = [r["method"] for r in rows]
    gf = [r["gflops"] for r in rows]
    best = gf.index(max(gf))
    colors = ["#295E96"] * len(gf)
    colors[best] = "#C7000B"
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, gf, color=colors)
    for b, g in zip(bars, gf):
        plt.text(
            b.get_x() + b.get_width() / 2,
            g,
            f"{g:.1f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("GFLOPS")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# 创建源代码目录
!mkdir -p src_gemm

## 5. v1 · 串行基准实现

与实验一、实验二相同，第一个版本包含**两个函数体相同**的串行实现，区别仅在于是否允许编译器自动向量化：

<!--
| 函数 | 说明 |
|---|---|
| `gemm_serial_no_vec` | 通过 `no-tree-vectorize` **显式关闭**自动向量化，作为**性能基准**（1.00×） |
| `gemm_serial` | 源码相同，但**允许编译器自动向量化** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>gemm_serial_no_vec</code></td>
      <td style="text-align: left;">通过 <code>no-tree-vectorize</code> <strong>显式关闭</strong>自动向量化，作为<strong>性能基准</strong>（1.00×）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>gemm_serial</code></td>
      <td style="text-align: left;">源码相同，但<strong>允许编译器自动向量化</strong></td>
    </tr>
  </tbody>
</table>

### 💡 关注点：编译器能自动向量化 GEMM 吗
实验二中我们看到，编译器面对**浮点规约**会退缩（`Serial (Auto)` 甚至比基准慢 2%）。GEMM 的内层 `k` 循环同样是规约，但它外面还套着两层循环，且 `B[k*N+j]` 是跨步访问。请在运行后观察 `Serial (Auto)` 一行的加速比——它接近 1.00× 吗？第 11 节会给出汇编层面的解释，以及加上 `-ffast-math` 后的变化。

### 数据布局与代码要点
- `A`：M×K 矩阵；`B`：K×N 矩阵；`C`：M×N 结果矩阵。均按**行主序**（row-major）存储。
- 参考结果 `C_ref` 由 `gemm_serial_no_vec` 生成；这一次调用同时完成了 A、B 的首次访问（page fault 与预热），因此不计入计时。
- **每次计时前都会 `memset(C_test, 0, bytes_C)`**：六个版本共用同一块结果缓冲区，若不清零，上一个版本留下的正确结果会掩盖下一个版本的错误——极端情况下，一个"什么都不做"的核函数也会被判为 PASS。`memset` 位于 `get_time_ms()` 之前，不进入计时区间。
- **`check_result` 采用随累加长度 K 缩放的相对容差**：$\text{tol}=2\times10^{-8}\,K\,\max|C_{ref}|$。float32 串行累加自身的误差随 K 近似线性增长，**固定的绝对容差在 K 增大后会把完全正确的结果误判为 FAIL**
- 计时口径：优化版本重复 `NTIMES = 5` 次取平均；串行基准单次即需数秒，故单独设 `NTIMES_BASE = 1`。

In [ ]:
%%writefile src_gemm/gemm_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run already takes seconds)

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The float32 serial accumulation itself loses precision roughly linearly in K.
// Measured on this data set:
//   max|serial - NEON| / (K * max|C_ref|) <= 2.9e-9   for K = 128..2048
// so tol = 2e-8 * K * max|C_ref| leaves a 7x..28x margin.
// A FIXED ABSOLUTE tolerance MUST NOT be used here: once K grows it reports a
// bogus FAIL on results that are perfectly correct.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0 to avoid inf)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// 1. Serial version (compiler may auto-vectorize)
// ---------------------------------------------------------
void gemm_serial(const float* restrict A, const float* restrict B,
                 float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 2. Serial version (auto-vectorization forced off) - performance baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

int main(int argc, char** argv) {
  if (argc != 4) {
    printf("Usage: %s <M> <N> <K>\n", argv[0]);
    printf("Example: %s 1024 1024 1024\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMM v1: Serial Baseline vs Auto-Vectorized Serial (C = A * B)\n");
  printf(" Matrix: A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Loops:  %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Serial, auto-vectorization allowed
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_auto = (end - start) / NTIMES_BASE;
  report("Serial (Auto)", t_auto, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/gemm_v1.c", "src_gemm/gemm_v1")
out_v1 = run_bin(BIN, 1024, 1024, 1024)

## 6. v2 · 朴素 NEON（1×4 微内核）

在 v1 的基础上**新增 `gemm_neon_naive` 函数**：把最内层的 `j` 循环向量化，一次算出 C 的 **4 个相邻元素**。

```c
for (j = 0; j <= N - 4; j += 4) {
  float32x4_t c_vec = vdupq_n_f32(0.0f);        // 4 个 C 元素的累加器
  for (k = 0; k < K; k++) {
    float32x4_t a_val = vdupq_n_f32(A[i*K+k]);  // 广播 1 个 A 标量
    float32x4_t b_vec = vld1q_f32(&B[k*N+j]);   // 加载 4 个连续的 B 元素
    c_vec = vfmaq_f32(c_vec, a_val, b_vec);     // 4 路乘加
  }
  vst1q_f32(&C[i*N+j], c_vec);
}
```

### 💡 关注点：为什么沿 j 而不是沿 k
- 沿 **$j$** 向量化：4 条通道对应 $C_{i,j}\dots C_{i,j+3}$ 四个**不同**元素，彼此独立；B 的地址 `B[k*N+j..j+3]` 是**连续的 16 字节**；循环结束直接写回，**不需要水平规约**。
- 沿 **$k$**（规约方向）向量化：4 条通道是同一个 $C_{ij}$ 的部分和，结尾必须做一次 `vaddvq_f32`；而且 B 的地址步长是 N，需要 4 条独立的标量加载再拼装成向量。

In [ ]:
%%writefile src_gemm/gemm_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run already takes seconds)

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The float32 serial accumulation itself loses precision roughly linearly in K.
// Measured on this data set:
//   max|serial - NEON| / (K * max|C_ref|) <= 2.9e-9   for K = 128..2048
// so tol = 2e-8 * K * max|C_ref| leaves a 7x..28x margin.
// A FIXED ABSOLUTE tolerance MUST NOT be used here: once K grows it reports a
// bogus FAIL on results that are perfectly correct.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0 to avoid inf)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling: when a dimension is not a multiple of 4, the remaining strip
// is finished with scalar code.
//   _store version overwrites (C  = sum): for versions that are not blocked
//                                          along K
//   _accum version accumulates (C += sum): for K-blocked versions, where each
//                                          K block only produces a partial sum
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial version (compiler may auto-vectorize)
// ---------------------------------------------------------
void gemm_serial(const float* restrict A, const float* restrict B,
                 float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 2. Serial version (auto-vectorization forced off) - performance baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. Naive NEON: 1x4 micro-kernel
//    Vectorize the innermost j loop: compute 4 adjacent elements of C at once.
//    Per k: broadcast 1 scalar of A + load 4 elements of B -> 1 FMA
//    Arithmetic intensity = 8 FLOP / 20 Byte = 0.4 FLOP/Byte
//    (still heavily memory bound)
// ---------------------------------------------------------
void gemm_neon_naive(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_vec = vdupq_n_f32(0.0f);
      for (int k = 0; k < K; k++) {
        float32x4_t a_val = vdupq_n_f32(A[i * K + k]);  // broadcast A[i][k]
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);   // load B[k][j..j+3]
        c_vec = vfmaq_f32(c_vec, a_val, b_vec);         // fused multiply-add
      }
      vst1q_f32(&C[i * N + j], c_vec);
    }
    // Column remainder (N not a multiple of 4)
    if (j < N) gemm_edge_store(1, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
}

int main(int argc, char** argv) {
  if (argc != 4) {
    printf("Usage: %s <M> <N> <K>\n", argv[0]);
    printf("Example: %s 1024 1024 1024\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMM v2: + Naive NEON 1x4 Micro-kernel (C = A * B)\n");
  printf(" Matrix: A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Loops:  %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Serial, auto-vectorization allowed
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_auto = (end - start) / NTIMES_BASE;
  report("Serial (Auto)", t_auto, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive NEON: 1x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_naive(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_naive = (end - start) / NTIMES;
  report("NEON Naive 1x4", t_naive, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/gemm_v2.c", "src_gemm/gemm_v2")
out_v2 = run_bin(BIN, 1024, 1024, 1024)

## 7. v3 · 寄存器分块（4×4 微内核）

在 v2 的基础上**新增 `gemm_neon_reg4x4` 函数**：一次计算 C 的一个 **4×4 子块**，用 4 个向量寄存器承载 16 个累加器。

```c
float32x4_t c_0, c_1, c_2, c_3;                 // 4 个累加器 = C 的 4 行
for (k = 0; k < K; k++) {
  float32x4_t b_vec = vld1q_f32(&B[k*N+j]);     // B 只加载【一次】
  c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i+0)*K+k]), b_vec);   // 被 4 行复用
  c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i+1)*K+k]), b_vec);
  c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i+2)*K+k]), b_vec);
  c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i+3)*K+k]), b_vec);
}
```

### 💡 关注点：复用从哪里来
v2 中，每加载一个 `b_vec` 只做 1 条 FMA；v3 中，**同一个 `b_vec` 被 4 行共用，做 4 条 FMA**。并且还**打破依赖链**：v2 只有 1 个累加器，`c_vec` 每次迭代都依赖上一次的结果，必须等满一个 FMA 延迟（约 4–5 周期）才能继续；v3 的 `c_0 / c_1 / c_2 / c_3` **相互独立**，4 条 FMA 可以在同一个延迟窗口里并行发射。

$$I=\frac{4\times4}{2(4+4)}=1.0\ \text{FLOP/Byte}\quad(\text{是 v2 的 }2.5\text{ 倍})$$

从指令层面看这个变化更直观：

<!--
| 版本 | 内层循环指令数 | `fmla` 条数 | 访存指令 | 栈溢出 | **fmla / 指令** |
|---|---|---|---|---|---|
| v2 · 1×4 | 6 | 1 | 2 | 0 | 0.17 |
| **v3 · 4×4** | **13** | **4** | **5** | **0** | **0.31** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">内层循环指令数</th>
      <th style="text-align: left;"><code>fmla</code> 条数</th>
      <th style="text-align: left;">访存指令</th>
      <th style="text-align: left;">栈溢出</th>
      <th style="text-align: left;"><strong>fmla / 指令</strong></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">v2 · 1×4</td>
      <td style="text-align: left;">6</td>
      <td style="text-align: left;">1</td>
      <td style="text-align: left;">2</td>
      <td style="text-align: left;">0</td>
      <td style="text-align: left;">0.17</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3 · 4×4</strong></td>
      <td style="text-align: left;"><strong>13</strong></td>
      <td style="text-align: left;"><strong>4</strong></td>
      <td style="text-align: left;"><strong>5</strong></td>
      <td style="text-align: left;"><strong>0</strong></td>
      <td style="text-align: left;"><strong>0.31</strong></td>
    </tr>
  </tbody>
</table>

同样多的循环控制开销（`cmp`/`add`/`bne`）被摊到 4 条 FMA 上，浮点单元的占空比接近翻倍。

> **寄存器是存储层次中最快的一级，而且完全由你支配。** 4×4 只用了 5 个向量寄存器（4 个累加器 + 1 个 B 向量），AArch64 的 32 个寄存器还剩大量余量——这正是扩展实验要做的事。

### ⚠️ 注意边界处理
M 或 N 不是 4 的倍数时，右侧与底部会剩下窄条。代码用 `gemm_edge_store` 以标量补齐。这段代码看似不重要，**但它是最容易出错、也最容易被"蒙混过关"的地方**——请务必用非 4 倍数的规模（如 `1023 1023 1023`）验证，见动手练习第 2 题。

In [ ]:
%%writefile src_gemm/gemm_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run already takes seconds)

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The float32 serial accumulation itself loses precision roughly linearly in K.
// Measured on this data set:
//   max|serial - NEON| / (K * max|C_ref|) <= 2.9e-9   for K = 128..2048
// so tol = 2e-8 * K * max|C_ref| leaves a 7x..28x margin.
// A FIXED ABSOLUTE tolerance MUST NOT be used here: once K grows it reports a
// bogus FAIL on results that are perfectly correct.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0 to avoid inf)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling: when a dimension is not a multiple of 4, the remaining strip
// is finished with scalar code.
//   _store version overwrites (C  = sum): for versions that are not blocked
//                                          along K
//   _accum version accumulates (C += sum): for K-blocked versions, where each
//                                          K block only produces a partial sum
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial version (compiler may auto-vectorize)
// ---------------------------------------------------------
void gemm_serial(const float* restrict A, const float* restrict B,
                 float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 2. Serial version (auto-vectorization forced off) - performance baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. Naive NEON: 1x4 micro-kernel
//    Vectorize the innermost j loop: compute 4 adjacent elements of C at once.
//    Per k: broadcast 1 scalar of A + load 4 elements of B -> 1 FMA
//    Arithmetic intensity = 8 FLOP / 20 Byte = 0.4 FLOP/Byte
//    (still heavily memory bound)
// ---------------------------------------------------------
void gemm_neon_naive(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_vec = vdupq_n_f32(0.0f);
      for (int k = 0; k < K; k++) {
        float32x4_t a_val = vdupq_n_f32(A[i * K + k]);  // broadcast A[i][k]
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);   // load B[k][j..j+3]
        c_vec = vfmaq_f32(c_vec, a_val, b_vec);         // fused multiply-add
      }
      vst1q_f32(&C[i * N + j], c_vec);
    }
    // Column remainder (N not a multiple of 4)
    if (j < N) gemm_edge_store(1, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
}

// ---------------------------------------------------------
// 4. Register blocking: 4x4 micro-kernel
//    Keep a 4x4 sub-block of C in 4 vector registers for the whole k loop.
//    Per k: load 4 scalars of A + 1 vector of B (4 elements) -> 4 FMAs
//    Arithmetic intensity = 32 FLOP / 32 Byte = 1.0 FLOP/Byte (2.5x over 1x4)
// ---------------------------------------------------------
void gemm_neon_reg4x4(const float* restrict A, const float* restrict B,
                      float* restrict C, int M, int N, int K) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      // 4 accumulators stay in vector registers; nothing is written to memory
      // during the whole k loop
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);  // B loaded only ONCE
        // The same b_vec is reused by all 4 rows -- this is where the data
        // reuse comes from
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * K + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * K + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * K + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * K + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * N + j], c_0);
      vst1q_f32(&C[(i + 1) * N + j], c_1);
      vst1q_f32(&C[(i + 2) * N + j], c_2);
      vst1q_f32(&C[(i + 3) * N + j], c_3);
    }
    // Column remainder
    if (j < N) gemm_edge_store(4, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
  // Row remainder
  if (i < M) gemm_edge_store(M - i, N, K, &A[i * K], K, B, N, &C[i * N], N);
}

int main(int argc, char** argv) {
  if (argc != 4) {
    printf("Usage: %s <M> <N> <K>\n", argv[0]);
    printf("Example: %s 1024 1024 1024\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMM v3: + Register Blocking 4x4 (C = A * B)\n");
  printf(" Matrix: A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Loops:  %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Serial, auto-vectorization allowed
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_auto = (end - start) / NTIMES_BASE;
  report("Serial (Auto)", t_auto, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive NEON: 1x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_naive(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_naive = (end - start) / NTIMES;
  report("NEON Naive 1x4", t_naive, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Register blocking: 4x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_reg4x4(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_reg = (end - start) / NTIMES;
  report("NEON Reg 4x4", t_reg, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/gemm_v3.c", "src_gemm/gemm_v3")
out_v3 = run_bin(BIN, 1024, 1024, 1024)

## 8. v4 · Cache 分块（loop tiling）

在 v3 的基础上**新增 `gemm_neon_tiled` 函数**：把 M、N、K 三个方向都切成 `BLOCK = 64` 的小块，微内核只在一个小块内工作。

```c
for (ii = 0; ii < M; ii += 64)          // C 的行块
  for (jj = 0; jj < N; jj += 64) {      // C 的列块
    clear C block;                      // ← 沿 K 累加前必须清零
    for (kk = 0; kk < K; kk += 64)      // K 方向切开
      microkernel_4x4(..., &A[ii*K+kk], &B[kk*N+jj], &C[ii*N+jj], ...);
  }
```

### 💡 关注点：分块尺寸怎么定
一个 64×64 的块占 $64\times64\times4=16$ KiB。微内核同时需要 A 块、B 块、C 块：

$$3\times16\ \text{KiB}=48\ \text{KiB}\;<\;\text{L1D}(64\ \text{KiB})$$

**块一旦装进 L1D，块内的所有复用就都是 L1 命中，不再回内存。** 这就是分块的全部意义。请注意分块**并不减少总的浮点运算量**，它只改变数据被访问的**次序**。

### ⚠️ 关键陷阱：覆盖写 → 累加写
K 方向被切开后，每个 K 块只算出**部分和**。因此微内核末尾必须由：

```c
vst1q_f32(&C[...], c_0);                                    // v3：覆盖写
```

改为：

```c
vst1q_f32(&C[...], vaddq_f32(vld1q_f32(&C[...]), c_0));     // v4：累加写
```

而这又要求：**进入 K 循环之前，必须把这个 C 块清零**（代码中的 `memset`）。同理，边界处理也要从 `gemm_edge_store`（`C = sum`）换成 `gemm_edge_accum`（`C += sum`）。

> 🐛 这是 GEMM 分块实现中最经典的一类 bug。忘记清零 → 结果里混进了上一次调用的残留；忘记改累加 → 只有最后一个 K 块的部分和被保留。两者都会让 `Check` 列报 FAIL。

In [ ]:
%%writefile src_gemm/gemm_v4.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run already takes seconds)

// Cache block sizes: one A block (BLOCK_M x BLOCK_K) + one B block (BLOCK_K x
// BLOCK_N) + one C block (BLOCK_M x BLOCK_N) = 3 x 64 x 64 x 4 B = 48 KiB,
// which fits in L1D/L2.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The float32 serial accumulation itself loses precision roughly linearly in K.
// Measured on this data set:
//   max|serial - NEON| / (K * max|C_ref|) <= 2.9e-9   for K = 128..2048
// so tol = 2e-8 * K * max|C_ref| leaves a 7x..28x margin.
// A FIXED ABSOLUTE tolerance MUST NOT be used here: once K grows it reports a
// bogus FAIL on results that are perfectly correct.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0 to avoid inf)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling: when a dimension is not a multiple of 4, the remaining strip
// is finished with scalar code.
//   _store version overwrites (C  = sum): for versions that are not blocked
//                                          along K
//   _accum version accumulates (C += sum): for K-blocked versions, where each
//                                          K block only produces a partial sum
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial version (compiler may auto-vectorize)
// ---------------------------------------------------------
void gemm_serial(const float* restrict A, const float* restrict B,
                 float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 2. Serial version (auto-vectorization forced off) - performance baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. Naive NEON: 1x4 micro-kernel
//    Vectorize the innermost j loop: compute 4 adjacent elements of C at once.
//    Per k: broadcast 1 scalar of A + load 4 elements of B -> 1 FMA
//    Arithmetic intensity = 8 FLOP / 20 Byte = 0.4 FLOP/Byte
//    (still heavily memory bound)
// ---------------------------------------------------------
void gemm_neon_naive(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_vec = vdupq_n_f32(0.0f);
      for (int k = 0; k < K; k++) {
        float32x4_t a_val = vdupq_n_f32(A[i * K + k]);  // broadcast A[i][k]
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);   // load B[k][j..j+3]
        c_vec = vfmaq_f32(c_vec, a_val, b_vec);         // fused multiply-add
      }
      vst1q_f32(&C[i * N + j], c_vec);
    }
    // Column remainder (N not a multiple of 4)
    if (j < N) gemm_edge_store(1, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
}

// ---------------------------------------------------------
// 4. Register blocking: 4x4 micro-kernel
//    Keep a 4x4 sub-block of C in 4 vector registers for the whole k loop.
//    Per k: load 4 scalars of A + 1 vector of B (4 elements) -> 4 FMAs
//    Arithmetic intensity = 32 FLOP / 32 Byte = 1.0 FLOP/Byte (2.5x over 1x4)
// ---------------------------------------------------------
void gemm_neon_reg4x4(const float* restrict A, const float* restrict B,
                      float* restrict C, int M, int N, int K) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      // 4 accumulators stay in vector registers; nothing is written to memory
      // during the whole k loop
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);  // B loaded only ONCE
        // The same b_vec is reused by all 4 rows -- this is where the data
        // reuse comes from
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * K + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * K + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * K + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * K + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * N + j], c_0);
      vst1q_f32(&C[(i + 1) * N + j], c_1);
      vst1q_f32(&C[(i + 2) * N + j], c_2);
      vst1q_f32(&C[(i + 3) * N + j], c_3);
    }
    // Column remainder
    if (j < N) gemm_edge_store(4, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
  // Row remainder
  if (i < M) gemm_edge_store(M - i, N, K, &A[i * K], K, B, N, &C[i * N], N);
}

// ---------------------------------------------------------
// 5. Cache blocking (loop tiling)
//    Cut M, N and K into blocks of 64 so that one block's working set fits in
//    L1D/L2.  The micro-kernel is exactly the same 4x4 kernel as in v3 with ONE
//    difference: once K is split, each K block only produces a PARTIAL sum, so
//    the result must be ACCUMULATED into C instead of overwriting it.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      // Accumulate into C (this block is only a partial sum along K)
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

void gemm_neon_tiled(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      // Clear this C block before accumulating the K blocks into it
      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

int main(int argc, char** argv) {
  if (argc != 4) {
    printf("Usage: %s <M> <N> <K>\n", argv[0]);
    printf("Example: %s 1024 1024 1024\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMM v4: + Cache Blocking / Loop Tiling (C = A * B)\n");
  printf(" Matrix: A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Loops:  %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Serial, auto-vectorization allowed
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_auto = (end - start) / NTIMES_BASE;
  report("Serial (Auto)", t_auto, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive NEON: 1x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_naive(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_naive = (end - start) / NTIMES;
  report("NEON Naive 1x4", t_naive, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Register blocking: 4x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_reg4x4(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_reg = (end - start) / NTIMES;
  report("NEON Reg 4x4", t_reg, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Cache blocking (loop tiling)
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_tiled(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/gemm_v4.c", "src_gemm/gemm_v4")
out_v4 = run_bin(BIN, 1024, 1024, 1024)

## 9. v5 · 内存打包（packing）

在 v4 的基础上**新增 `gemm_neon_packed` 函数**：在微内核开工之前，先把 B 的当前块**复制**到一段连续的缓冲区。

```c
pack_matrix_B(cur_K, cur_N, &B[kk*N+jj], N, packed_B);   // 先复制
microkernel_4x4_packed(..., packed_B, ...);              // 再计算
```

### 💡 关注点：为什么"多复制一遍"反而更快
v4 中，微内核访问 B 的地址是 `B[k*ldb + j]`，`ldb = N = 1024`：

<!--
| | v4（原地访问 B） | v5（访问打包后的 B） |
|---|---|---|
| 相邻两个 k 的地址差 | **4096 B**（跨页） | **256 B**（同页） |
| 一个 64×64 块跨越的页数 | **64 页** → 64 个 TLB 表项 | **4 页** → 4 个 TLB 表项 |
| 每条 64 B cache line 的利用率 | 16 B / 64 B = **25%** | **100%** |
| `vld1q_f32` 的地址对齐 | 取决于 `j` | 缓冲区 16 B 对齐 + 行距补齐到 4 个 float，**恒定 16 B 对齐** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">v4（原地访问 B）</th>
      <th style="text-align: left;">v5（访问打包后的 B）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">相邻两个 k 的地址差</td>
      <td style="text-align: left;"><strong>4096 B</strong>（跨页）</td>
      <td style="text-align: left;"><strong>256 B</strong>（同页）</td>
    </tr>
    <tr>
      <td style="text-align: left;">一个 64×64 块跨越的页数</td>
      <td style="text-align: left;"><strong>64 页</strong> → 64 个 TLB 表项</td>
      <td style="text-align: left;"><strong>4 页</strong> → 4 个 TLB 表项</td>
    </tr>
    <tr>
      <td style="text-align: left;">每条 64 B cache line 的利用率</td>
      <td style="text-align: left;">16 B / 64 B = <strong>25%</strong></td>
      <td style="text-align: left;"><strong>100%</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>vld1q_f32</code> 的地址对齐</td>
      <td style="text-align: left;">取决于 <code>j</code></td>
      <td style="text-align: left;">缓冲区 16 B 对齐 + 行距补齐到 4 个 float，<strong>恒定 16 B 对齐</strong></td>
    </tr>
  </tbody>
</table>

打包的代价是 $cur\_K\times cur\_N$ 次复制，而它服务的计算量是 $cur\_M\times cur\_K\times cur\_N$ 次乘加——**开销占比约 $1/cur\_M\approx1.6\%$**，很容易被上表的收益覆盖。

> **这是高性能 GEMM 库（OpenBLAS、BLIS、ARM Compute Library）的标准做法。** 它们甚至会把 A 也一并打包，并把数据重排成微内核需要的确切顺序——扩展实验就要做这件事。

### 📌 关于"补零"
`pack_matrix_B` 把每行补齐到 4 个 float 的整数倍（`N_aligned = (N + 3) & ~3`）。**补的零并不参与主循环的计算**（主循环只走到 `j <= N-4`），它的作用是让缓冲区的**行距**是 16 字节的整数倍，从而保证每一次 `vld1q_f32` 都落在 16 字节边界上。

In [ ]:
%%writefile src_gemm/gemm_v5.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run already takes seconds)

// Cache block sizes: one A block (BLOCK_M x BLOCK_K) + one B block (BLOCK_K x
// BLOCK_N) + one C block (BLOCK_M x BLOCK_N) = 3 x 64 x 64 x 4 B = 48 KiB,
// which fits in L1D/L2.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The float32 serial accumulation itself loses precision roughly linearly in K.
// Measured on this data set:
//   max|serial - NEON| / (K * max|C_ref|) <= 2.9e-9   for K = 128..2048
// so tol = 2e-8 * K * max|C_ref| leaves a 7x..28x margin.
// A FIXED ABSOLUTE tolerance MUST NOT be used here: once K grows it reports a
// bogus FAIL on results that are perfectly correct.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0 to avoid inf)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling: when a dimension is not a multiple of 4, the remaining strip
// is finished with scalar code.
//   _store version overwrites (C  = sum): for versions that are not blocked
//                                          along K
//   _accum version accumulates (C += sum): for K-blocked versions, where each
//                                          K block only produces a partial sum
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial version (compiler may auto-vectorize)
// ---------------------------------------------------------
void gemm_serial(const float* restrict A, const float* restrict B,
                 float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 2. Serial version (auto-vectorization forced off) - performance baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. Naive NEON: 1x4 micro-kernel
//    Vectorize the innermost j loop: compute 4 adjacent elements of C at once.
//    Per k: broadcast 1 scalar of A + load 4 elements of B -> 1 FMA
//    Arithmetic intensity = 8 FLOP / 20 Byte = 0.4 FLOP/Byte
//    (still heavily memory bound)
// ---------------------------------------------------------
void gemm_neon_naive(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_vec = vdupq_n_f32(0.0f);
      for (int k = 0; k < K; k++) {
        float32x4_t a_val = vdupq_n_f32(A[i * K + k]);  // broadcast A[i][k]
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);   // load B[k][j..j+3]
        c_vec = vfmaq_f32(c_vec, a_val, b_vec);         // fused multiply-add
      }
      vst1q_f32(&C[i * N + j], c_vec);
    }
    // Column remainder (N not a multiple of 4)
    if (j < N) gemm_edge_store(1, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
}

// ---------------------------------------------------------
// 4. Register blocking: 4x4 micro-kernel
//    Keep a 4x4 sub-block of C in 4 vector registers for the whole k loop.
//    Per k: load 4 scalars of A + 1 vector of B (4 elements) -> 4 FMAs
//    Arithmetic intensity = 32 FLOP / 32 Byte = 1.0 FLOP/Byte (2.5x over 1x4)
// ---------------------------------------------------------
void gemm_neon_reg4x4(const float* restrict A, const float* restrict B,
                      float* restrict C, int M, int N, int K) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      // 4 accumulators stay in vector registers; nothing is written to memory
      // during the whole k loop
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);  // B loaded only ONCE
        // The same b_vec is reused by all 4 rows -- this is where the data
        // reuse comes from
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * K + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * K + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * K + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * K + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * N + j], c_0);
      vst1q_f32(&C[(i + 1) * N + j], c_1);
      vst1q_f32(&C[(i + 2) * N + j], c_2);
      vst1q_f32(&C[(i + 3) * N + j], c_3);
    }
    // Column remainder
    if (j < N) gemm_edge_store(4, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
  // Row remainder
  if (i < M) gemm_edge_store(M - i, N, K, &A[i * K], K, B, N, &C[i * N], N);
}

// ---------------------------------------------------------
// 5. Cache blocking (loop tiling)
//    Cut M, N and K into blocks of 64 so that one block's working set fits in
//    L1D/L2.  The micro-kernel is exactly the same 4x4 kernel as in v3 with ONE
//    difference: once K is split, each K block only produces a PARTIAL sum, so
//    the result must be ACCUMULATED into C instead of overwriting it.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      // Accumulate into C (this block is only a partial sum along K)
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

void gemm_neon_tiled(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      // Clear this C block before accumulating the K blocks into it
      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

// ---------------------------------------------------------
// 6. Memory packing
//    In v4 the micro-kernel reads B at B[k*ldb + j] with ldb = N = 1024, so two
//    consecutive k values are 4 KiB apart: every k lands on a DIFFERENT physical
//    page.  A 64-deep K block therefore needs 64 TLB entries, and only
//    16 B out of every 64 B cache line is actually used.
//    Copying the B block into a contiguous buffer first turns the micro-kernel
//    into a purely sequential read:
//      - TLB entries drop from 64 to 4 (16 KiB / 4 KiB)
//      - every cache line is fully consumed
//      - the row stride is padded to a multiple of 4 floats (16 B), so every
//        vld1q_f32 is a 16-byte aligned access
//    Cost of packing: cur_K*cur_N copies against cur_M*cur_K*cur_N multiply-adds,
//    i.e. about 1/cur_M ~= 1.6% of the total work - easily paid back.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad to keep the
                                                        // stride aligned
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

void gemm_neon_packed(const float* restrict A, const float* restrict B,
                      float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

int main(int argc, char** argv) {
  if (argc != 4) {
    printf("Usage: %s <M> <N> <K>\n", argv[0]);
    printf("Example: %s 1024 1024 1024\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMM v5: + Memory Packing of B (C = A * B)\n");
  printf(" Matrix: A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Loops:  %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Serial, auto-vectorization allowed
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_auto = (end - start) / NTIMES_BASE;
  report("Serial (Auto)", t_auto, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive NEON: 1x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_naive(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_naive = (end - start) / NTIMES;
  report("NEON Naive 1x4", t_naive, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Register blocking: 4x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_reg4x4(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_reg = (end - start) / NTIMES;
  report("NEON Reg 4x4", t_reg, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Cache blocking (loop tiling)
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_tiled(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Memory packing of B
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_packed(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_packed = (end - start) / NTIMES;
  report("NEON Packed", t_packed, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/gemm_v5.c", "src_gemm/gemm_v5")
out_v5 = run_bin(BIN, 1024, 1024, 1024)

## 10. 📈 性能可视化（基于 v5 的六版本结果）

v5 的输出包含全部六个实现的耗时、加速比与实际算力，据此绘制两张图：**加速比**（相对串行基准）与 **GFLOPS**（绝对算力）。

（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

In [ ]:
rows_v5 = parse_table(out_v5)
for r in rows_v5:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:.2f}x')
plot_speedup(rows_v5, "GEMM: speedup of six implementations (1024x1024x1024)")
plot_gflops(parse_gflops(out_v5), "GEMM: achieved GFLOPS (1024x1024x1024)")

## 11. 结果分析

> 注：具体数值随硬件平台、矩阵规模、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

GEMM 的结果通常揭示以下几个规律：

---

**① `Serial (Auto)` 与基准几乎无差别（≈1.00×）——编译器根本没有向量化。**

这不是"向量化了但效果不好"，而是**一条向量指令都没生成**。GCC 9.4 的 `-fopt-info-vec-all` 对 `gemm_serial` 的三重循环报告：

```
missed: not vectorized: multiple nested loops.
missed: not vectorized: control flow in loop.
note: vectorized 0 loops in function.
```

生成的内层循环是纯标量：

```asm
.L55:
  ldr   s2, [x0], 4          // A[i][k]，连续
  ldr   s1, [x1]             // B[k][j]，跨步 N
  fmadd s0, s2, s1, s0       // 标量乘加，串行依赖链
  bne   .L55
```

**② 但加上 `-ffast-math` 之后，编译器"能"向量化了——而且它选错了方向。**

同一份源码加 `-ffast-math` 重新编译，GCC 报告 `loop vectorized using 16 byte vectors`，生成的内层循环变成：

```asm
.L59:
  ldr   s2, [x0, x7]         // ┐
  ldr   s0, [x0]             // │ 从 B 的一列里【逐个】取 4 个元素
  ldr   s4, [x0, x6]         // │ （步长 N，无法用一条向量加载）
  ldr   s3, [x0, x5]         // ┘
  add   x0, x0, x2, sxtw 4
  ins   v0.s[1], v2.s[0]     // ┐
  ldr   q2, [x1], 16         // │ 再用 3 条 ins 把它们拼成一个向量
  ins   v0.s[2], v4.s[0]     // │
  cmp   x1, x4               // │
  ins   v0.s[3], v3.s[0]     // ┘
  fmla  v1.4s, v2.4s, v0.4s  // ← 12 条指令，只换来 1 条 FMA
  bne   .L59
```

编译器沿 **$k$ 方向**（规约方向）向量化了。A 沿 k 连续，可以一条 `ldr q` 装 4 个；但 **B 沿 k 的步长是 N**，只能用 4 条标量 `ldr` + 3 条 `ins` 手工拼装。**12 条指令换 1 条 `fmla`**，比不向量化还糟。

对照我们手写的 v2（沿 **$j$ 方向**）：

```asm
.L76:  // 6 条指令，1 条 fmla —— 而且 B 是一条完整的向量加载
```

> **🎓 核心结论：向量化不是"打开开关"，而是"选对方向"。**
> 编译器只看得见循环的语法结构，看不见"哪个方向的访问是连续的、哪个方向没有依赖"。GEMM 的正确答案是沿 $j$ 方向——**这个判断必须由人来做**，这正是本实验必须手写 NEON 的根本原因。

---

**③ v2（1×4）的加速比非常接近 4×——这是"只用满了宽度"的标志。**

理论 SIMD 位宽是 4，实测也差不多接近 4。**意味着瓶颈还没有换。** 基准和 v2 都被同一件事卡住——FMA 的延迟没有被隐藏，访存延迟又被这条长依赖链顺带掩盖了。v2 既没有打破依赖链（只有 1 个累加器），也没有提高数据复用（$I=0.4$）。

> **向量化只是把"一次算几个"从 1 提到 4，别的什么都没改。**

---

**④ v3（4×4 寄存器分块）是第一次"实质性"的提升。**

算术强度由 0.4 升到 1.0（2.5 倍），指令层面的 `fmla` 占比由 0.17 升到 0.31（1.8 倍）。这两个数字大致框定了 v3 相对 v2 的提升幅度。

> **寄存器分块之所以有效，是因为它同时改善了两件事**：既减少了访存次数（B 向量被复用 4 次），又摊薄了循环控制开销。

---

**⑤ v4（Cache 分块）的收益取决于工作集是否超出缓存。**

默认规模 1024³ 下，三个矩阵各 4 MiB、合计 12 MiB。v3 的微内核在整个 K=1024 上连续跑，扫过 B 的一整列（跨越 4 MiB 地址范围），**远超 L1D(64 KiB) 与 L2(512 KiB)**，几乎每次访问都要回到 L3 甚至内存。v4 把工作集压到 48 KiB，块内复用全部变成 L1 命中。

> 📏 **先量噪声，再下结论。** 参照实验一的经验，同一份代码连续跑多次，耗时散布可达 5–15%。如果 v4 相对 v3 的提升在 10% 以内，请先重复运行 3–5 次确认它超出了噪声，再去解释它。

---

**⑥ v5（内存打包）的收益主要来自 TLB 与 cache line 利用率，幅度通常小于 v3、v4。**

打包解决的是"访问模式"问题而非"数据量"问题：cache line 利用率由 25% 升到 100%，TLB 表项由 64 个降到 4 个。如果本机的 TLB 较大、或硬件预取器足够聪明，v5 相对 v4 的提升可能很小甚至为负（打包的 1.6% 开销没有被覆盖）。**这本身就是一个值得分析的结果**——它说明这台机器的瓶颈已经不在 B 的访问模式上了。

---

**⑦ 六个版本合计通常能到 3–8×，但离峰值仍有很大距离。**

不妨算一笔账：若单核有 2 条 128 位 FMA 流水线、主频 $f$，峰值算力 $=2\times4\times2\times f=16f$ FLOP/s（2.6 GHz 约 41.6 GFLOPS）。对照图中的 GFLOPS 柱子，v5 达到峰值的百分之几？

4×4 微内核的天花板可以直接算出来：内层循环 13 条指令里只有 4 条 `fmla`，即使每周期发射 4 条指令，也需要 3.25 个周期才能发完 4 条 FMA——**约合每周期 1.2 条 FMA，只有双流水线峰值的 60%**。而 8×8 微内核是 27 条指令 16 条 `fmla`，每周期约 2 条 FMA，**理论上可以打满**。这就是扩展实验的意义。

---

### 🎓 结论

GEMM 揭示了 SIMD 编程的完整图景：**优化不是一招，而是一条针对存储层次逐级展开的链条。**

<!--
| 优化 | 针对的存储层次 | 手段 | 本实验版本 |
|---|---|---|---|
| 向量化 | 寄存器（宽度） | 一条指令算 4 个 | v2 |
| 寄存器分块 | 寄存器（数量） | 数据在寄存器里被复用 | v3 |
| Cache 分块 | L1D / L2 | 让工作集装得下 | v4 |
| 内存打包 | TLB / cache line | 让访问变连续 | v5 |
| 更大的微内核 | 寄存器（用满） | 把算术强度推到 2.0 | 🚀 扩展实验 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">优化</th>
      <th style="text-align: left;">针对的存储层次</th>
      <th style="text-align: left;">手段</th>
      <th style="text-align: left;">本实验版本</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">向量化</td>
      <td style="text-align: left;">寄存器（宽度）</td>
      <td style="text-align: left;">一条指令算 4 个</td>
      <td style="text-align: left;">v2</td>
    </tr>
    <tr>
      <td style="text-align: left;">寄存器分块</td>
      <td style="text-align: left;">寄存器（数量）</td>
      <td style="text-align: left;">数据在寄存器里被复用</td>
      <td style="text-align: left;">v3</td>
    </tr>
    <tr>
      <td style="text-align: left;">Cache 分块</td>
      <td style="text-align: left;">L1D / L2</td>
      <td style="text-align: left;">让工作集装得下</td>
      <td style="text-align: left;">v4</td>
    </tr>
    <tr>
      <td style="text-align: left;">内存打包</td>
      <td style="text-align: left;">TLB / cache line</td>
      <td style="text-align: left;">让访问变连续</td>
      <td style="text-align: left;">v5</td>
    </tr>
    <tr>
      <td style="text-align: left;">更大的微内核</td>
      <td style="text-align: left;">寄存器（用满）</td>
      <td style="text-align: left;">把算术强度推到 2.0</td>
      <td style="text-align: left;">🚀 扩展实验</td>
    </tr>
  </tbody>
</table>

**每一级的收益都是有上限的，而且必须逐级打通**：不做寄存器分块，Cache 分块也救不了访存瓶颈；不做 Cache 分块，打包省下的 TLB 也无处发力。这就是为什么高性能库的 GEMM 实现动辄数千行——它们把这条链条的每一环都拧到了极致。

## 12. 🚀 扩展实验：8×8 双打包微内核

> **本节是留给学生独立完成的挑战。** 下面的代码框架已经写好了全部脚手架（分块循环、B 的打包、边界处理、计时与校验），只留下**两个 `TODO`** 需要你补全。补全前运行会得到 `FAIL`，补全后应当得到 `PASS`，并且成为表格中最快的一行。

### 🎯 目标

把微内核从 4×4 扩大到 **8×8**，并把 **A 也打包**，使算术强度由 1.0 提高到 **2.0 FLOP/Byte**。

$$I_{8\times8}=\frac{8\times8}{2(8+8)}=2.0\ \text{FLOP/Byte}$$

### 🧩 设计要点

**(1) 为什么 A 也必须打包。**
4×4 微内核里，A 的 4 个标量地址是 `A[(i+0..3)*lda + k]`，彼此相差 `lda` 个 float——4 条**互不相邻**的标量加载。扩到 8 行就是 8 条。把 A 按"8 行一条、条内按 k 递增、每个 k 连续存放 8 个值"重排之后，微内核里读 A 就变成**纯顺序的 `*a_ptr++`**，编译器可以合并成 `ldp`（成对加载）。

**(2) 打包后的数据布局。**

```
A 的 8×K 子块              打包缓冲区（按 k 递增，每 k 连续 8 个 float）
  a00 a01 a02 ...          [a00 a10 a20 a30 a40 a50 a60 a70]  ← k=0
  a10 a11 a12 ...    -->   [a01 a11 a21 a31 a41 a51 a61 a71]  ← k=1
  ...                      [a02 a12 a22 a32 a42 a52 a62 a72]  ← k=2
  a70 a71 a72 ...          ...
```

B 的打包完全对称（按 8 列一条），代码中 `pack_B_panel_8` **已经写好，请把它当作 `pack_A_panel_8` 的范本**。

**(3) 微内核结构。**
需要 **16 个累加器**（8 行 × 每行 2 个向量 = 8×8 个 float），加上 2 个 B 向量，共 18 个向量寄存器 —— AArch64 的 32 个够用，不会溢出到栈。内层每个 k：

```c
float32x4_t b0 = vld1q_f32(b_ptr);        // B 的第 0..3 列
float32x4_t b1 = vld1q_f32(b_ptr + 4);    // B 的第 4..7 列
b_ptr += 8;
float a0 = *a_ptr++;                      // A 的第 0 行
c00 = vfmaq_n_f32(c00, b0, a0);           // C[0][0..3] += a0 * b0
c01 = vfmaq_n_f32(c01, b1, a0);           // C[0][4..7] += a0 * b1
// ... 对 8 行各来一遍，共 16 条 FMA
```

**(4) 循环次序。**
框架采用 GotoBLAS 的经典次序 `jj → kk → ii`：B 块每 `(jj,kk)` 只打包一次，在整个 `ii` 循环中复用；A 块在 `ii` 内打包。

### ✅ 验收标准

1. `Check` 列为 **PASS**；
2. 用 `1023 1023 1023`、`130 133 137` 等非 8 倍数规模验证边界，仍为 **PASS**；
3. `NEON 8x8 Pack` 成为表格中**最快**的一行；
4. 用 `gcc -O3 -S` 反汇编，确认 `microkernel_8x8_packed` 的内层循环里**没有** `[sp, ...]` 形式的访存（即没有寄存器溢出）。

### 💭 完成后请思考

- 8×8 相比 4×4，算术强度翻倍，但实测提速通常达不到 2×。差距去了哪里？
- 试着把微内核改成 8×12（24 个累加器）。性能是继续上升还是下降？用反汇编解释。
- 本实验全程是**单核**的。如果再加上 OpenMP 多线程，应该并行化哪一层循环？为什么不能并行化 `k` 循环？

In [ ]:
%%writefile src_gemm/gemm_ext.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 5      // Repetitions for the optimized versions
#define NTIMES_BASE 1 // Repetitions for the serial baseline (one run already takes seconds)

// Cache block sizes: one A block (BLOCK_M x BLOCK_K) + one B block (BLOCK_K x
// BLOCK_N) + one C block (BLOCK_M x BLOCK_N) = 3 x 64 x 64 x 4 B = 48 KiB,
// which fits in L1D/L2.
#define BLOCK_M 64
#define BLOCK_N 64
#define BLOCK_K 64

#define MIN(a, b) ((a) < (b) ? (a) : (b))

// ---------------------------------------------------------
// Timing and result checking
// ---------------------------------------------------------
double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Relative tolerance, SCALED WITH the reduction length K.
// The float32 serial accumulation itself loses precision roughly linearly in K.
// Measured on this data set:
//   max|serial - NEON| / (K * max|C_ref|) <= 2.9e-9   for K = 128..2048
// so tol = 2e-8 * K * max|C_ref| leaves a 7x..28x margin.
// A FIXED ABSOLUTE tolerance MUST NOT be used here: once K grows it reports a
// bogus FAIL on results that are perfectly correct.
const char* check_result(const float* restrict ref, const float* restrict test,
                         int M, int N, int K) {
  double max_diff = 0.0, max_ref = 0.0;
  for (size_t i = 0; i < (size_t)M * (size_t)N; i++) {
    double diff = fabs((double)ref[i] - (double)test[i]);
    if (diff > max_diff) max_diff = diff;
    if (fabs((double)ref[i]) > max_ref) max_ref = fabs((double)ref[i]);
  }
  double tol = 2e-8 * (double)K * (max_ref > 1.0 ? max_ref : 1.0);
  return (max_diff <= tol) ? "PASS" : "FAIL";
}

// Print one row of the report table (skip the division when t <= 0 to avoid inf)
void report(const char* name, double t, double t_base, double ops,
            const char* chk) {
  if (t <= 0.0) {
    printf("| %-15s | %9.3f |     -   |      - |  %-4s |\n", name, t, chk);
  } else {
    printf("| %-15s | %9.3f | %5.2f x | %6.2f |  %-4s |\n", name, t,
           t_base / t, ops / (t * 1e6), chk);
  }
}

// ---------------------------------------------------------
// Edge handling: when a dimension is not a multiple of 4, the remaining strip
// is finished with scalar code.
//   _store version overwrites (C  = sum): for versions that are not blocked
//                                          along K
//   _accum version accumulates (C += sum): for K-blocked versions, where each
//                                          K block only produces a partial sum
// ---------------------------------------------------------
void gemm_edge_store(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] = sum;
    }
  }
}

void gemm_edge_accum(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) sum += A[i * lda + k] * B[k * ldb + j];
      C[i * ldc + j] += sum;
    }
  }
}

// ---------------------------------------------------------
// 1. Serial version (compiler may auto-vectorize)
// ---------------------------------------------------------
void gemm_serial(const float* restrict A, const float* restrict B,
                 float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 2. Serial version (auto-vectorization forced off) - performance baseline
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void gemm_serial_no_vec(const float* restrict A, const float* restrict B, float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    for (int j = 0; j < N; j++) {
      float sum = 0.0f;
      for (int k = 0; k < K; k++) {
        sum += A[i * K + k] * B[k * N + j];
      }
      C[i * N + j] = sum;
    }
  }
}

// ---------------------------------------------------------
// 3. Naive NEON: 1x4 micro-kernel
//    Vectorize the innermost j loop: compute 4 adjacent elements of C at once.
//    Per k: broadcast 1 scalar of A + load 4 elements of B -> 1 FMA
//    Arithmetic intensity = 8 FLOP / 20 Byte = 0.4 FLOP/Byte
//    (still heavily memory bound)
// ---------------------------------------------------------
void gemm_neon_naive(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int i = 0; i < M; i++) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_vec = vdupq_n_f32(0.0f);
      for (int k = 0; k < K; k++) {
        float32x4_t a_val = vdupq_n_f32(A[i * K + k]);  // broadcast A[i][k]
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);   // load B[k][j..j+3]
        c_vec = vfmaq_f32(c_vec, a_val, b_vec);         // fused multiply-add
      }
      vst1q_f32(&C[i * N + j], c_vec);
    }
    // Column remainder (N not a multiple of 4)
    if (j < N) gemm_edge_store(1, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
}

// ---------------------------------------------------------
// 4. Register blocking: 4x4 micro-kernel
//    Keep a 4x4 sub-block of C in 4 vector registers for the whole k loop.
//    Per k: load 4 scalars of A + 1 vector of B (4 elements) -> 4 FMAs
//    Arithmetic intensity = 32 FLOP / 32 Byte = 1.0 FLOP/Byte (2.5x over 1x4)
// ---------------------------------------------------------
void gemm_neon_reg4x4(const float* restrict A, const float* restrict B,
                      float* restrict C, int M, int N, int K) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      // 4 accumulators stay in vector registers; nothing is written to memory
      // during the whole k loop
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * N + j]);  // B loaded only ONCE
        // The same b_vec is reused by all 4 rows -- this is where the data
        // reuse comes from
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * K + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * K + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * K + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * K + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * N + j], c_0);
      vst1q_f32(&C[(i + 1) * N + j], c_1);
      vst1q_f32(&C[(i + 2) * N + j], c_2);
      vst1q_f32(&C[(i + 3) * N + j], c_3);
    }
    // Column remainder
    if (j < N) gemm_edge_store(4, N - j, K, &A[i * K], K, &B[j], N, &C[i * N + j], N);
  }
  // Row remainder
  if (i < M) gemm_edge_store(M - i, N, K, &A[i * K], K, B, N, &C[i * N], N);
}

// ---------------------------------------------------------
// 5. Cache blocking (loop tiling)
//    Cut M, N and K into blocks of 64 so that one block's working set fits in
//    L1D/L2.  The micro-kernel is exactly the same 4x4 kernel as in v3 with ONE
//    difference: once K is split, each K block only produces a PARTIAL sum, so
//    the result must be ACCUMULATED into C instead of overwriting it.
// ---------------------------------------------------------
void microkernel_4x4(int M, int N, int K, const float* A, int lda,
                     const float* B, int ldb, float* C, int ldc) {
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B[k * ldb + j]);
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      // Accumulate into C (this block is only a partial sum along K)
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B, ldb, &C[i * ldc], ldc);
}

void gemm_neon_tiled(const float* restrict A, const float* restrict B,
                     float* restrict C, int M, int N, int K) {
  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      // Clear this C block before accumulating the K blocks into it
      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        microkernel_4x4(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                        &B[kk * N + jj], N, &C[ii * N + jj], N);
      }
    }
  }
}

// ---------------------------------------------------------
// 6. Memory packing
//    In v4 the micro-kernel reads B at B[k*ldb + j] with ldb = N = 1024, so two
//    consecutive k values are 4 KiB apart: every k lands on a DIFFERENT physical
//    page.  A 64-deep K block therefore needs 64 TLB entries, and only
//    16 B out of every 64 B cache line is actually used.
//    Copying the B block into a contiguous buffer first turns the micro-kernel
//    into a purely sequential read:
//      - TLB entries drop from 64 to 4 (16 KiB / 4 KiB)
//      - every cache line is fully consumed
//      - the row stride is padded to a multiple of 4 floats (16 B), so every
//        vld1q_f32 is a 16-byte aligned access
//    Cost of packing: cur_K*cur_N copies against cur_M*cur_K*cur_N multiply-adds,
//    i.e. about 1/cur_M ~= 1.6% of the total work - easily paid back.
// ---------------------------------------------------------
void pack_matrix_B(int K, int N, const float* B, int ldb, float* buffer) {
  int N_aligned = (N + 3) & ~3;  // round the row stride up to 4 floats
  float* ptr = buffer;
  for (int k = 0; k < K; k++) {
    for (int j = 0; j < N; j++) *ptr++ = B[k * ldb + j];
    for (int j = N; j < N_aligned; j++) *ptr++ = 0.0f;  // pad to keep the
                                                        // stride aligned
  }
}

void microkernel_4x4_packed(int M, int N, int K, const float* A, int lda,
                            const float* B_packed, float* C, int ldc) {
  int ldb = (N + 3) & ~3;  // row stride of the packed buffer
  int i = 0;
  for (; i <= M - 4; i += 4) {
    int j = 0;
    for (; j <= N - 4; j += 4) {
      float32x4_t c_0 = vdupq_n_f32(0.0f);
      float32x4_t c_1 = vdupq_n_f32(0.0f);
      float32x4_t c_2 = vdupq_n_f32(0.0f);
      float32x4_t c_3 = vdupq_n_f32(0.0f);

      for (int k = 0; k < K; k++) {
        float32x4_t b_vec = vld1q_f32(&B_packed[k * ldb + j]);  // sequential
        c_0 = vfmaq_f32(c_0, vdupq_n_f32(A[(i + 0) * lda + k]), b_vec);
        c_1 = vfmaq_f32(c_1, vdupq_n_f32(A[(i + 1) * lda + k]), b_vec);
        c_2 = vfmaq_f32(c_2, vdupq_n_f32(A[(i + 2) * lda + k]), b_vec);
        c_3 = vfmaq_f32(c_3, vdupq_n_f32(A[(i + 3) * lda + k]), b_vec);
      }
      vst1q_f32(&C[(i + 0) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 0) * ldc + j]), c_0));
      vst1q_f32(&C[(i + 1) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 1) * ldc + j]), c_1));
      vst1q_f32(&C[(i + 2) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 2) * ldc + j]), c_2));
      vst1q_f32(&C[(i + 3) * ldc + j],
                vaddq_f32(vld1q_f32(&C[(i + 3) * ldc + j]), c_3));
    }
    if (j < N) gemm_edge_accum(4, N - j, K, &A[i * lda], lda, &B_packed[j], ldb, &C[i * ldc + j], ldc);
  }
  if (i < M) gemm_edge_accum(M - i, N, K, &A[i * lda], lda, B_packed, ldb, &C[i * ldc], ldc);
}

void gemm_neon_packed(const float* restrict A, const float* restrict B,
                      float* restrict C, int M, int N, int K) {
  int ldb_pack = (BLOCK_N + 3) & ~3;
  float* packed_B =
      (float*)aligned_alloc(16, (size_t)BLOCK_K * ldb_pack * sizeof(float));
  if (!packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    return;
  }

  for (int ii = 0; ii < M; ii += BLOCK_M) {
    int cur_M = MIN(BLOCK_M, M - ii);
    for (int jj = 0; jj < N; jj += BLOCK_N) {
      int cur_N = MIN(BLOCK_N, N - jj);

      for (int r = 0; r < cur_M; r++)
        memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));

      for (int kk = 0; kk < K; kk += BLOCK_K) {
        int cur_K = MIN(BLOCK_K, K - kk);
        pack_matrix_B(cur_K, cur_N, &B[kk * N + jj], N, packed_B);
        microkernel_4x4_packed(cur_M, cur_N, cur_K, &A[ii * K + kk], K,
                               packed_B, &C[ii * N + jj], N);
      }
    }
  }
  free(packed_B);
}

// ---------------------------------------------------------
// 7. [EXTENSION LAB] 8x8 double-packed micro-kernel
//    (1) BOTH A and B are packed: A into strips of 8 rows, B into strips of
//        8 columns, both laid out k-major, so the micro-kernel reads both of
//        them purely sequentially.
//    (2) The kernel computes 8x8 = 64 elements of C at once and needs 16 vector
//        accumulators.  AArch64 has 32 vector registers, so there is still
//        headroom and GCC does not spill to the stack.
//    (3) Arithmetic intensity = 128 FLOP / 64 Byte = 2.0 FLOP/Byte, twice 4x4.
//    Loop order follows GotoBLAS (jc -> pc -> ic): a B block is packed once per
//    (jj,kk) pair and reused across the whole ii loop.
// ---------------------------------------------------------
#define EXT_BLOCK_M 128
#define EXT_BLOCK_N 128
#define EXT_BLOCK_K 128

// Pack an (M x K) block of A into strips of 8 rows.
// Within a strip: k ascending, and for each k the 8 row values are contiguous.
void pack_A_panel_8(int K, int M, const float* A, int lda, float* buffer) {
  // ============================ TODO 1 ============================
  // Following the pack_B_panel_8 below as a template, pack the (M x K) block of
  // A into strips of 8 rows:
  //   the outer i loop advances 8 rows at a time;
  //   within a strip, k ascends;
  //   for each k write 8 contiguous floats: A[i+0][k], A[i+1][k], ..., A[i+7][k];
  //   write 0.0f whenever i + r >= M (fewer than 8 rows left).
  // Hint: three nested loops whose body is a single "*buffer++ = ...;"
  // The line below is a placeholder (it just zeroes the buffer) and makes the
  // result FAIL.  Delete it and write the real implementation.
  memset(buffer, 0, (size_t)((M + 7) / 8 * 8) * K * sizeof(float));
  (void)A;
  (void)lda;
  // ================================================================
}

// Pack a (K x N) block of B into strips of 8 columns.
// Within a strip: k ascending, and for each k the 8 column values are contiguous.
void pack_B_panel_8(int K, int N, const float* B, int ldb, float* buffer) {
  for (int j = 0; j < N; j += 8) {
    for (int k = 0; k < K; k++) {
      for (int c = 0; c < 8; c++)
        *buffer++ = (j + c < N) ? B[k * ldb + (j + c)] : 0.0f;  // pad if < 8 cols
    }
  }
}

// 8x8 micro-kernel: C[0..7][0..7] += A_panel * B_panel
void microkernel_8x8_packed(int K, const float* A_panel, const float* B_panel,
                            float* C, int ldc) {
  float32x4_t c00 = vdupq_n_f32(0.0f), c01 = vdupq_n_f32(0.0f);
  float32x4_t c10 = vdupq_n_f32(0.0f), c11 = vdupq_n_f32(0.0f);
  float32x4_t c20 = vdupq_n_f32(0.0f), c21 = vdupq_n_f32(0.0f);
  float32x4_t c30 = vdupq_n_f32(0.0f), c31 = vdupq_n_f32(0.0f);
  float32x4_t c40 = vdupq_n_f32(0.0f), c41 = vdupq_n_f32(0.0f);
  float32x4_t c50 = vdupq_n_f32(0.0f), c51 = vdupq_n_f32(0.0f);
  float32x4_t c60 = vdupq_n_f32(0.0f), c61 = vdupq_n_f32(0.0f);
  float32x4_t c70 = vdupq_n_f32(0.0f), c71 = vdupq_n_f32(0.0f);

  const float* a_ptr = A_panel;
  const float* b_ptr = B_panel;

  for (int k = 0; k < K; k++) {
    float32x4_t b0 = vld1q_f32(b_ptr);      // 8 columns of B = 2 vectors
    float32x4_t b1 = vld1q_f32(b_ptr + 4);
    b_ptr += 8;
    __builtin_prefetch(b_ptr + 64, 0, 3);   // prefetch ahead, keep in L1

    // ============================ TODO 2 ============================
    // For each of the 8 rows: take the next scalar of A from a_ptr, then do one
    // vfmaq_n_f32 against b0 and one against b1, accumulating into that row's
    // two accumulators.  8 groups, 16 FMAs in total.
    // Row 0 is given; write rows 1..7 the same way (and delete the (void) line).
    float a0 = *a_ptr++;  c00 = vfmaq_n_f32(c00, b0, a0);  c01 = vfmaq_n_f32(c01, b1, a0);
    (void)c10; (void)c11; (void)c20; (void)c21; (void)c30; (void)c31;
    (void)c40; (void)c41; (void)c50; (void)c51; (void)c60; (void)c61;
    (void)c70; (void)c71;
    // ================================================================
  }

  vst1q_f32(C + 0 * ldc + 0, vaddq_f32(vld1q_f32(C + 0 * ldc + 0), c00));
  vst1q_f32(C + 0 * ldc + 4, vaddq_f32(vld1q_f32(C + 0 * ldc + 4), c01));
  vst1q_f32(C + 1 * ldc + 0, vaddq_f32(vld1q_f32(C + 1 * ldc + 0), c10));
  vst1q_f32(C + 1 * ldc + 4, vaddq_f32(vld1q_f32(C + 1 * ldc + 4), c11));
  vst1q_f32(C + 2 * ldc + 0, vaddq_f32(vld1q_f32(C + 2 * ldc + 0), c20));
  vst1q_f32(C + 2 * ldc + 4, vaddq_f32(vld1q_f32(C + 2 * ldc + 4), c21));
  vst1q_f32(C + 3 * ldc + 0, vaddq_f32(vld1q_f32(C + 3 * ldc + 0), c30));
  vst1q_f32(C + 3 * ldc + 4, vaddq_f32(vld1q_f32(C + 3 * ldc + 4), c31));
  vst1q_f32(C + 4 * ldc + 0, vaddq_f32(vld1q_f32(C + 4 * ldc + 0), c40));
  vst1q_f32(C + 4 * ldc + 4, vaddq_f32(vld1q_f32(C + 4 * ldc + 4), c41));
  vst1q_f32(C + 5 * ldc + 0, vaddq_f32(vld1q_f32(C + 5 * ldc + 0), c50));
  vst1q_f32(C + 5 * ldc + 4, vaddq_f32(vld1q_f32(C + 5 * ldc + 4), c51));
  vst1q_f32(C + 6 * ldc + 0, vaddq_f32(vld1q_f32(C + 6 * ldc + 0), c60));
  vst1q_f32(C + 6 * ldc + 4, vaddq_f32(vld1q_f32(C + 6 * ldc + 4), c61));
  vst1q_f32(C + 7 * ldc + 0, vaddq_f32(vld1q_f32(C + 7 * ldc + 0), c70));
  vst1q_f32(C + 7 * ldc + 4, vaddq_f32(vld1q_f32(C + 7 * ldc + 4), c71));
}

void gemm_neon_8x8(const float* restrict A, const float* restrict B,
                   float* restrict C, int M, int N, int K) {
  size_t szA = (size_t)EXT_BLOCK_K * (EXT_BLOCK_M + 8) * sizeof(float);
  size_t szB = (size_t)EXT_BLOCK_K * (EXT_BLOCK_N + 8) * sizeof(float);
  float* packed_A = (float*)aligned_alloc(16, szA);
  float* packed_B = (float*)aligned_alloc(16, szB);
  if (!packed_A || !packed_B) {
    printf("Error: packed buffer allocation failed.\n");
    free(packed_A);
    free(packed_B);
    return;
  }

  for (int jj = 0; jj < N; jj += EXT_BLOCK_N) {
    int cur_N = MIN(EXT_BLOCK_N, N - jj);
    for (int kk = 0; kk < K; kk += EXT_BLOCK_K) {
      int cur_K = MIN(EXT_BLOCK_K, K - kk);

      // The B block is packed once per (jj,kk) and reused across the ii loop
      pack_B_panel_8(cur_K, cur_N, &B[kk * N + jj], N, packed_B);

      for (int ii = 0; ii < M; ii += EXT_BLOCK_M) {
        int cur_M = MIN(EXT_BLOCK_M, M - ii);

        if (kk == 0) {  // first K block: clear C first
          for (int r = 0; r < cur_M; r++)
            memset(&C[(ii + r) * N + jj], 0, (size_t)cur_N * sizeof(float));
        }

        pack_A_panel_8(cur_K, cur_M, &A[ii * K + kk], K, packed_A);

        for (int i = 0; i < cur_M; i += 8) {
          const float* ap = packed_A + (size_t)i * cur_K;
          for (int j = 0; j < cur_N; j += 8) {
            const float* bp = packed_B + (size_t)j * cur_K;
            if (i + 8 <= cur_M && j + 8 <= cur_N) {
              microkernel_8x8_packed(cur_K, ap, bp, &C[(ii + i) * N + (jj + j)],
                                     N);
            } else {
              // Edge: still read from the packed data (the zero padding is
              // harmless because it contributes 0 to the sum)
              for (int r = 0; r < 8 && i + r < cur_M; r++) {
                for (int c = 0; c < 8 && j + c < cur_N; c++) {
                  float sum = 0.0f;
                  for (int k = 0; k < cur_K; k++)
                    sum += ap[k * 8 + r] * bp[k * 8 + c];
                  C[(ii + i + r) * N + (jj + j + c)] += sum;
                }
              }
            }
          }
        }
      }
    }
  }
  free(packed_A);
  free(packed_B);
}

int main(int argc, char** argv) {
  if (argc != 4) {
    printf("Usage: %s <M> <N> <K>\n", argv[0]);
    printf("Example: %s 1024 1024 1024\n", argv[0]);
    return 1;
  }

  int M = atoi(argv[1]);
  int N = atoi(argv[2]);
  int K = atoi(argv[3]);
  if (M <= 0 || N <= 0 || K <= 0) return 1;

  printf("============================================================\n");
  printf(" GEMM ext: + 8x8 Double-Packed Micro-kernel (STUDENT VERSION)\n");
  printf(" Matrix: A(%d x %d) * B(%d x %d) = C(%d x %d)\n", M, K, K, N, M, N);
  printf(" Loops:  %d  (baseline: %d)\n", NTIMES, NTIMES_BASE);
  printf("============================================================\n");

  // aligned_alloc requires size to be a multiple of the alignment; round to 16 B
  size_t bytes_A = (((size_t)M * K * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_B = (((size_t)K * N * sizeof(float)) + 15) & ~(size_t)15;
  size_t bytes_C = (((size_t)M * N * sizeof(float)) + 15) & ~(size_t)15;

  float* A = (float*)aligned_alloc(16, bytes_A);
  float* B = (float*)aligned_alloc(16, bytes_B);
  float* C_ref = (float*)aligned_alloc(16, bytes_C);   // Golden result
  float* C_test = (float*)aligned_alloc(16, bytes_C);  // Reusable buffer

  if (!A || !B || !C_ref || !C_test) {
    printf("Alloc failed\n");
    return 1;
  }

  // Initialization: different moduli for A and B to avoid degenerate periodic data
  for (int i = 0; i < M; i++)
    for (int k = 0; k < K; k++) A[(size_t)i * K + k] = (float)((i + k) % 100) * 0.001f;
  for (int k = 0; k < K; k++)
    for (int j = 0; j < N; j++) B[(size_t)k * N + j] = (float)((k + j) % 97) * 0.001f;

  // Golden reference = serial baseline (this call also first-touches A and B)
  gemm_serial_no_vec(A, B, C_ref, M, N, K);

  double ops = 2.0 * (double)M * (double)N * (double)K;
  double start, end;

  printf("\n--------------------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | GFLOPS | Check |\n");
  printf("|-----------------|-----------|---------|--------|-------|\n");

  // Baseline: serial, no vectorization
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial_no_vec(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_base = (end - start) / NTIMES_BASE;
  report("Serial (No-Vec)", t_base, t_base, ops, "-");

  // Serial, auto-vectorization allowed
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES_BASE; t++) gemm_serial(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_auto = (end - start) / NTIMES_BASE;
  report("Serial (Auto)", t_auto, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Naive NEON: 1x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_naive(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_naive = (end - start) / NTIMES;
  report("NEON Naive 1x4", t_naive, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Register blocking: 4x4 micro-kernel
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_reg4x4(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_reg = (end - start) / NTIMES;
  report("NEON Reg 4x4", t_reg, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Cache blocking (loop tiling)
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_tiled(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_tiled = (end - start) / NTIMES;
  report("NEON Tiled", t_tiled, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // Memory packing of B
  memset(C_test, 0, bytes_C);   // Clear: otherwise the previous version's
                                // correct result would mask this one's error
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_packed(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_packed = (end - start) / NTIMES;
  report("NEON Packed", t_packed, t_base, ops, check_result(C_ref, C_test, M, N, K));

  // [EXTENSION LAB] 8x8 double-packed micro-kernel
  memset(C_test, 0, bytes_C);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) gemm_neon_8x8(A, B, C_test, M, N, K);
  end = get_time_ms();
  double t_8x8 = (end - start) / NTIMES;
  report("NEON 8x8 Pack", t_8x8, t_base, ops, check_result(C_ref, C_test, M, N, K));

  printf("--------------------------------------------------------------\n");

  free(A);
  free(B);
  free(C_ref);
  free(C_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_gemm/gemm_ext.c", "src_gemm/gemm_ext")
out_ext = run_bin(BIN, 1024, 1024, 1024)

# 补全 TODO 之前，最后一行应当是 FAIL；补全之后应当 PASS 且成为最快的一行。
rows_ext = parse_table(out_ext)
if rows_ext:
    plot_speedup(rows_ext, "GEMM + 8x8 double-packed kernel (1024x1024x1024)")
    plot_gflops(parse_gflops(out_ext), "GEMM + 8x8: achieved GFLOPS")

## 13. 🔧 动手练习

请修改代码、重新编译并运行，观察性能的变化（建议先独立完成，再阅读思考题）：

1. **先量噪声。** 把 v5 单元格连续运行 3–5 次，记录每个版本耗时的极差。参照实验一的经验，散布可能达到 5–15%。此后凡是小于噪声的差异，一律不要为它编故事。

2. **边界正确性。** 分别以 `1023 1023 1023`、`1025 1025 1025`、`130 133 137` 运行 v5。所有 `Check` 是否仍为 PASS？如果把 `gemm_neon_reg4x4` 末尾的 `if (i < M) gemm_edge_store(...)` 改成 `if (0) ...`，哪些规模会 FAIL、哪些不会？（这一题揭示了"用 4 的倍数测试"为什么远远不够。）

3. **校验的可信度。** 在 v5 的 `gemm_neon_naive` 函数体开头加一行 `return;`，让它什么都不做，重新编译运行。`Check` 列报什么？再把该版本计时前的那句 `memset(C_test, 0, bytes_C);` 注释掉，重新运行——这次报什么？（提示：不清零时，上一个版本留下的正确结果会掩盖本版本的错误，一个"空转"的核函数也能报 PASS，同时打印出上万倍的加速比。）

4. **分块尺寸的影响。** 把 v4/v5 的 `BLOCK_M/BLOCK_N/BLOCK_K` 由 64 依次改为 32、96、128、256，记录性能曲线。在哪个值附近达到最优？用 `lscpu -C`（或 `cat /sys/devices/system/cpu/cpu0/cache/index*/size`）查出本机的 L1D 与 L2 容量，验证"三个块合计应装进 L1D"这一估算。

5. **规模相关性。** 分别以 `512 512 512`、`1024 1024 1024`、`2048 2048 2048` 运行 v5（注意 2048 的串行基准会很慢，可临时把 `NTIMES_BASE` 保持为 1）。观察 v4/v5 相对 v3 的优势如何随规模变化。在哪个规模下 Cache 分块几乎没有收益？为什么？

6. **【进阶】编译器的向量化方向。** 用 `gcc -O3 -fPIC -ffast-math -fopt-info-vec src_gemm/gemm_v1.c -o /tmp/a -lm` 重新编译，观察 `Serial (Auto)` 一行的变化；再用 `gcc -O3 -fPIC -ffast-math -S src_gemm/gemm_v1.c -o -` 反汇编，找出 `gemm_serial` 的内层循环，数一数其中有多少条 `ins` 指令、多少条 `fmla`。这与第 11 节 ② 的分析是否一致？⚠️ 注意 `-ffast-math` 会改变浮点语义（允许重排加法），生产代码中需谨慎使用。

7. **【进阶】寄存器溢出的直接证据。** 用 `gcc -O3 -fPIC -S src_gemm/gemm_v5.c -o -` 反汇编，分别找出 `microkernel_4x4` 与（完成扩展实验后的）`microkernel_8x8_packed` 的内层循环，统计每个循环里 `fmla` 的条数与总指令数之比。再把 8×8 改成 16×16（64 个累加器）重新反汇编，数一数内层循环里出现了多少条访问 `[sp, ...]` 的指令。

## 14. 🤔 思考题

- GEMM 的计算量是 $O(n^3)$、访存量是 $O(n^2)$，为什么朴素实现的性能反而只有峰值的百分之几？"理论上可以复用"与"实际复用了"之间差的是什么？
- 为什么手写 NEON 选择沿 $j$ 方向向量化，而编译器（在 `-ffast-math` 下）选择沿 $k$ 方向？两者在**访存连续性**与**数据依赖**上各有什么代价？为什么编译器做不出正确的选择？
- 微内核尺寸 $m\times n$ 的算术强度是 $\dfrac{mn}{2(m+n)}$。既然越大越好，为什么不直接用 32×32？限制它的物理量是什么？
- 沿 K 方向分块之后，为什么微内核必须由"覆盖写"改成"累加写"？如果忘记在 K 循环前清零 C 块，会观察到什么现象？如果忘记改成累加写呢？
- 内存打包多做了一次完整的数据复制，为什么反而更快？它改善的是**数据量**还是**访问模式**？在什么情况下打包会变成净亏损？
- 本实验的 `check_result` 采用随 K 缩放的相对容差 $2\times10^{-8}K\max|C_{ref}|$。为什么不能像实验一那样用一个固定的绝对容差？如果把矩阵元素的初始化由 `(i+k)%100 * 0.001f` 改成 `rand()/(float)RAND_MAX`（量级放大约 1000 倍），容差公式还成立吗？
- 每次计时前的 `memset(C_test, 0, bytes_C)` 为什么是必须的？它会不会污染计时结果？如果把它移到 `get_time_ms()` 之后会怎样？
- 五级优化中，哪一级的收益**最依赖于矩阵规模**？哪一级**几乎与规模无关**？

## 15. 小结与后续

本实验完成了 GEMM 从朴素三重循环到接近工业级实现的完整优化链条：

<!--
| 版本 | 新增内容 | 算术强度 | 涉及知识点 |
|---|---|---|---|
| **v1** | `gemm_serial_no_vec` + `gemm_serial` | 0.25 | 性能基准、编译器为何不敢向量化三重循环 |
| **v2** | `gemm_neon_naive` | 0.40 | 向量化方向的选择、广播 + FMA 范式 |
| **v3** | `gemm_neon_reg4x4` | 1.00 | 寄存器分块、数据复用、算术强度 |
| **v4** | `gemm_neon_tiled` | 1.00 | Cache 分块、工作集、覆盖写 → 累加写 |
| **v5** | `gemm_neon_packed` | 1.00 | 内存打包、TLB、cache line 利用率、地址对齐 |
| **🚀 扩展** | `gemm_neon_8x8` | 2.00 | 双打包、16 累加器、寄存器数量的硬约束 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">算术强度</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>gemm_serial_no_vec</code> + <code>gemm_serial</code></td>
      <td style="text-align: left;">0.25</td>
      <td style="text-align: left;">性能基准、编译器为何不敢向量化三重循环</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>gemm_neon_naive</code></td>
      <td style="text-align: left;">0.40</td>
      <td style="text-align: left;">向量化方向的选择、广播 + FMA 范式</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>gemm_neon_reg4x4</code></td>
      <td style="text-align: left;">1.00</td>
      <td style="text-align: left;">寄存器分块、数据复用、算术强度</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v4</strong></td>
      <td style="text-align: left;"><code>gemm_neon_tiled</code></td>
      <td style="text-align: left;">1.00</td>
      <td style="text-align: left;">Cache 分块、工作集、覆盖写 → 累加写</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v5</strong></td>
      <td style="text-align: left;"><code>gemm_neon_packed</code></td>
      <td style="text-align: left;">1.00</td>
      <td style="text-align: left;">内存打包、TLB、cache line 利用率、地址对齐</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>🚀 扩展</strong></td>
      <td style="text-align: left;"><code>gemm_neon_8x8</code></td>
      <td style="text-align: left;">2.00</td>
      <td style="text-align: left;">双打包、16 累加器、寄存器数量的硬约束</td>
    </tr>
  </tbody>
</table>

通过 GEMM，我们把第三章的三条主线汇合到了一起：

1. **实验一 AXPY** 教会我们：逐元素运算的瓶颈是**带宽**，向量化的天花板由算术强度决定；
2. **实验二 GEMV** 教会我们：**规约**打断了向量化，而**打破依赖**与**数据复用**是两类不同的优化；
3. **本实验 GEMM** 则说明：当算术强度足够高时，性能瓶颈可以从内存**一路让回到浮点单元**——但这需要针对存储层次的**每一级**都做对应的优化，缺一不可。

> **🎓 最重要的一句话**：优化不是"用了 SIMD 就快了"，而是**把数据搬运的次数降到理论下界附近**。向量化、寄存器分块、Cache 分块、内存打包——四件事做的是同一件事，只是分别作用在存储层次的不同层级上。